In [7]:
import numpy as np

import torch
from torch.utils.data import Dataset
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

#### The ```Dataset``` class


In PyTorch, a dataset is represented by a regular Python class that inherits from the Dataset class. You can think of it as a list of tuples, each tuple corresponding to one point (features, label).

- ```__init__(self)```: It takes whatever arguments are needed to build a list of tuples; it may be the name of a CSV file that will be loaded and processed; it may be two tensors with one for features and another one for labels; or anything else, depending on the task at hand.

----------------------------

There is no need to load the whole dataset in the constructor method (```__init__```). If your dataset is big (e.g., tens of thousands of image files), loading it at once would not be memory efficient. It is recommended to load them on demand whenever ```__get_item__``` is called.

----------------------------


- ```__get_item__(self, index)```: It allows the dataset to be indexed so that it can work like a list (```dataset[i]```). It should return a tuple (features, label) corresponding to the requested data point. We can either return the corresponding slices of our pre-loaded dataset or, as mentioned above, load them on demand (like in this example).

- ```__len__(self)```: It should simply return the size of the whole dataset. So, whenever it is sampled, its indexing is limited to the actual size.

In [9]:
true_b = 1
true_w = 2
N = 100

# Data generation process
np.random.seed(42)
x = np.random.rand(N, 1)
epsilon = (.1 * np.random.randn(N, 1))
y = true_b + true_w * x + epsilon

# Shuffles the indices
idx = np.arange(N)
np.random.shuffle(idx)

# Uses first 80 random indices for train
train_idx = idx[:int(N*.8)]
val_idx = idx[int(N*.8):]

x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]

print('x_train: {}'.format(x_train[:5]))

x_train: [[0.77127035]
 [0.06355835]
 [0.86310343]
 [0.02541913]
 [0.73199394]]


In [10]:
class CustomDataset(Dataset):
    def __init__(self, x_tensor, y_tensor):
        self.x = x_tensor
        self.y = y_tensor
        
    def __getitem__(self, index):
        return (self.x[index], self.y[index])

    def __len__(self):
        return len(self.x)

# Wait, is this a CPU tensor now? Why? Where is .to(device)?
x_train_tensor = torch.as_tensor(x_train).float()
y_train_tensor = torch.as_tensor(y_train).float()

train_data = CustomDataset(x_train_tensor, y_train_tensor)
print(train_data[0])

(tensor([0.7713]), tensor([2.4745]))


### Tensor Dataset

In [11]:
train_data = TensorDataset(x_train_tensor, y_train_tensor)
print(train_data[0])

(tensor([0.7713]), tensor([2.4745]))


### We use it for mini-batch GD


There is more to a ```DataLoader``` than meets the eye. For instance, it is also possible to use it together with a sampler to fetch mini-batches that compensate for imbalanced classes.

In [12]:
train_loader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)

In [13]:
# will return a list -> 2 tensors (features, labels)
print(next(iter(train_loader)))

[tensor([[0.1818],
        [0.0740],
        [0.0055],
        [0.2921],
        [0.6011],
        [0.8084],
        [0.2912],
        [0.4952],
        [0.7713],
        [0.9395],
        [0.5924],
        [0.2809],
        [0.1159],
        [0.3117],
        [0.9219],
        [0.7069]]), tensor([[1.3734],
        [1.1713],
        [1.0632],
        [1.5848],
        [2.1214],
        [2.6141],
        [1.4361],
        [1.8735],
        [2.4745],
        [2.8890],
        [2.1687],
        [1.5846],
        [1.1603],
        [1.7637],
        [2.8506],
        [2.4388]])]


### Use the dataset and data loader in mini-batch

In [ ]:
############################################################
############################################################
#####
#####      From Before:
#####
def make_train_step(model, loss_fn, optimizer):
    # Builds function that performs a step in the train loop
    def perform_train_step(x, y):
        # Sets model to TRAIN mode
        model.train()
        
        # Step 1 - computes model's predictions - forward pass
        yhat = model(x)
        # Step 2 - computes the loss
        loss = loss_fn(yhat, y)
        # Step 3 - computes gradients for "b" and "w" parameters
        loss.backward()
        # Step 4 - updates parameters using gradients and the learning rate
        optimizer.step()
        optimizer.zero_grad()
        
        # Returns the loss
        return loss.item()
    
    # Returns the function that will be called inside the 
    # train loop
    return perform_train_step

def make_val_step(model, loss_fn):
    # Builds function that performs a step in the validation loop
    def perform_val_step(x, y):
        # Sets model to EVAL mode
        model.eval()     # 1)
        
        # Step 1 - Computes our model's predicted output
        yhat = model(x) # forward pass
        # Step 2 - Computes the loss
        loss = loss_fn(yhat, y)
        # There is no need to compute Steps 3 and 4, 
        # since we don't update parameters during evaluation
        return loss.item()
    
    return perform_val_step

############################################################
device = 'cuda' if torch.cuda.is_available() else 'cpu'
lr = 0.1
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(1, 1)).to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)
loss_fn = nn.MSELoss(reduction='mean')
train_step = make_train_step(model, loss_fn, optimizer) # 1)

In [ ]:
n_epochs = 1000

losses = []
for epoch in range(n_epochs):
    # inner loop
    mini_batch_losses = []                              
    for x_batch, y_batch in train_loader:
        # the dataset "lives" in the CPU, so do our mini-batches
        # therefore, we need to send those mini-batches to the
        # device where the model "lives"
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        # Performs one train step and returns the 
        # corresponding loss for this mini-batch
        mini_batch_loss = train_step(x_batch, y_batch)
        mini_batch_losses.append(mini_batch_loss)

    # Computes average loss over all mini-batches (That's the epoch loss)
    loss = np.mean(mini_batch_losses)
    losses.append(loss)

In [ ]:
def mini_batch(device, data_loader, step):
    mini_batch_losses = []
    for x_batch, y_batch in data_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        mini_batch_loss = step(x_batch, y_batch)
        mini_batch_losses.append(mini_batch_loss)

    loss = np.mean(mini_batch_losses)
    return loss

###########################################################################
###########################################################################
n_epochs = 200
losses = []
for epoch in range(n_epochs):
    # inner loop
    loss = mini_batch(device, train_loader, train_step) # 1)
    losses.append(loss)